# Are VLAs secretly kNN?

A lookup over the demos predicts the VLAs' action chunks, and their chunks sit closer to the training data than a new human demonstration does.

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import utils as u

FS_SMALL, FS_TINY = 9, 7  # pt: \small, \scriptsize of the paper
PAGE_WIDTH = 5.5  # in: \textwidth of the paper (\linewidth of a figure)
ROOT = Path.cwd().parent
data_path = ROOT / "results"
figs_path = ROOT / "analysis/figs"
style_path = ROOT / "analysis/figure.mplstyle"

In [ ]:
data = json.loads((data_path / "data.json").read_text())
figs_path.mkdir(exist_ok=True)
plt.style.use(style_path)

In [ ]:
rows = u.lookup_rows(data)
y = np.arange(len(rows))[::-1]
human = float(np.mean([r[2] for r in rows]))
labels = [re.sub(r"π([\d.]+)", r"$\\pi_{\1}$", r[0]).replace("†", r"$^\dagger$") for r in rows]
FIGSIZE = (0.48 * PAGE_WIDTH, 0.612 + 0.1944 * len(rows))  # both figures side by side, each at 0.48\linewidth
rows

## Both panels in one figure

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(PAGE_WIDTH, 0.612 + 0.1944 * len(rows)), sharey=True, layout="constrained")
for yi, (_lab, r2, _d, ratio) in zip(y, rows, strict=True):
    ax1.barh(yi, r2, color=u.Colour.LEARNED, height=0.62, edgecolor="white")
    ax1.text(r2 + 0.02, yi, f"{r2:.2f}", va="center", fontsize=FS_TINY, bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.5})
    ax2.barh(yi, ratio, color=u.Colour.LEARNED, height=0.62, edgecolor="white")
    ax2.text(ratio + 0.02, yi, f"{ratio:.2f}", va="center", fontsize=FS_TINY, bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.5})
ax1.axvline(human, color=u.Colour.KNN, linestyle=(0, (4, 3)), linewidth=1.6)
ax1.text(human - 0.01, len(rows) - 0.45, "Human demos", ha="right", va="bottom", fontsize=FS_TINY, color=u.Colour.KNN)
ax2.axvline(1.0, color=u.Colour.KNN, linestyle=(0, (4, 3)), linewidth=1.6)
ax2.text(0.99, len(rows) - 0.45, "Human demos", ha="right", va="bottom", fontsize=FS_TINY, color=u.Colour.KNN)
ax1.set_xlim(0, 1.12)
ax2.set_xlim(0, 1.35)
ax1.set_xlabel("$R^2$ of a lookup's prediction", fontsize=FS_SMALL)
ax2.set_xlabel("distance to nearest training chunk (÷ human)", fontsize=FS_SMALL)
for ax in (ax1, ax2):
    ax.tick_params(axis="y", length=0)
    ax.spines["left"].set_visible(False)
    ax.xaxis.grid(True)
    ax.set_axisbelow(True)
    ax.tick_params(axis="x", labelsize=FS_TINY)
ax1.set_yticks(y, labels)
ax1.set_ylim(-0.6, len(rows) + 0.3)
fig.savefig(figs_path / "vla_lookup.pdf")

## Two separate figures (`figure.mplstyle`)

### (a) Lookup prediction R²

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE, layout="constrained")
for yi, (_lab, r2, _d, _ratio) in zip(y, rows, strict=True):
    ax.barh(yi, r2, color=u.Colour.LEARNED, height=0.62, edgecolor="white")
    ax.text(r2 + 0.02, yi, f"{r2:.2f}", va="center", fontsize=FS_TINY, bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.5})# fmt: skip
ax.axvline(human, color=u.Colour.KNN, linestyle=(0, (4, 3)), linewidth=1.6)
ax.text(human - 0.03, len(rows) - 0.45, "Ground-truth chunks", ha="right", va="bottom", fontsize=FS_TINY,
        color=u.Colour.KNN, bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.5})  # fmt: skip
ax.set_xlim(0, 1.12)
ax.set_ylim(-0.6, len(rows) + 0.3)
ax.set_yticks(y, labels)
ax.set_xlabel("$R^2$ to the $k$-NN prediction", fontsize=FS_SMALL)
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.tick_params(axis="x", labelsize=FS_TINY)
fig.savefig(figs_path / "vla_lookup_r2.pdf")

### (b) Distance to nearest training chunk

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE, layout="constrained")
for yi, (_lab, _r2, _d, ratio) in zip(y, rows, strict=True):
    ax.barh(yi, ratio, color=u.Colour.LEARNED, height=0.62, edgecolor="white")
    ax.text(ratio + 0.02, yi, f"{ratio:.2f}", va="center", fontsize=FS_TINY, bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.5})# fmt: skip
ax.axvline(1.0, color=u.Colour.KNN, linestyle=(0, (4, 3)), linewidth=1.6)
ax.text(0.97, len(rows) - 0.45, "Ground-truth chunks", ha="right", va="bottom", fontsize=FS_TINY,
        color=u.Colour.KNN, bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.5})  # fmt: skip
ax.set_xlim(0, 1.35)
ax.set_xticks([0, 0.5, 1])
ax.set_ylim(-0.6, len(rows) + 0.3)
ax.set_yticks(y, labels)
ax.set_xlabel("Squared error to nearest training\nchunk, relative to ground truth", fontsize=FS_SMALL)
ax.tick_params(axis="y", length=0)
ax.spines["left"].set_visible(False)
ax.xaxis.grid(True)
ax.set_axisbelow(True)
ax.tick_params(axis="x", labelsize=FS_TINY)
fig.savefig(figs_path / "vla_lookup_distance.pdf")